# 실습과제 Q5: MLP 품질예측

**관련 차시**: 22차시 - 딥러닝 실습: MLP 품질예측  
**난이도**: ★★★☆☆ (중급)

---

## 학습 목표

1. StandardScaler로 데이터 정규화하기
2. Keras Sequential 모델로 MLP 구축하기
3. 과적합 방지 기법(Dropout, EarlyStopping) 적용하기
4. 학습 곡선을 해석하고 모델 성능 개선하기

In [ ]:
# 필요한 라이브러리 임포트
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

# TensorFlow/Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# 재현성을 위한 시드 설정
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow 버전: {tf.__version__}")

---

## 문제 1: 데이터 준비 및 정규화 (20점)

데이터를 로드하고 **StandardScaler**로 정규화하세요.

In [ ]:
# 데이터 로드
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00601/ai4i2020.csv"
df = pd.read_csv(url)

# 특성과 타겟 분리
feature_cols = ['Air temperature [K]', 'Process temperature [K]',
                'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
X = df[feature_cols]
y = df['Machine failure']

# 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"학습 데이터: {X_train.shape}")
print(f"테스트 데이터: {X_test.shape}")

In [ ]:
# TODO: StandardScaler 적용 (빈칸 채우기)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # 학습 데이터: fit + transform
X_test_scaled = scaler.______(X_test)  # 테스트 데이터: transform만 (fit 없이)

# 정규화 전후 비교
print("=== 정규화 전 (X_train 첫 5행) ===")
print(X_train.head())
print(f"\n정규화 전 평균: {X_train.mean().values.round(2)}")
print(f"정규화 전 표준편차: {X_train.std().values.round(2)}")

In [ ]:
print("\n=== 정규화 후 (X_train_scaled 첫 5행) ===")
print(X_train_scaled[:5].round(3))
print(f"\n정규화 후 평균: {X_train_scaled.mean(axis=0).round(4)}")
print(f"정규화 후 표준편차: {X_train_scaled.std(axis=0).round(4)}")

### 문제 1 답변 작성

- 정규화 전 평균: ______________________
- 정규화 후 평균: ______________________
- StandardScaler가 하는 일: ______________________
- 신경망에서 정규화가 중요한 이유: ______________________
- `fit_transform`과 `transform`의 차이점: ______________________

---

## 문제 2: MLP 모델 설계 (25점)

Keras Sequential API를 사용하여 **MLP 모델**을 설계하세요.

**모델 구조**:
- 입력층: 5개 특성
- 은닉층 1: 32개 뉴런, ReLU 활성화
- Dropout: 30%
- 은닉층 2: 16개 뉴런, ReLU 활성화
- 출력층: 1개 뉴런, Sigmoid 활성화

In [ ]:
# TODO: 모델 정의 (빈칸 채우기)
model = Sequential([
    Dense(32, activation='relu', input_shape=(5,)),
    Dropout(______),  # 0.3 (30% 드롭아웃)
    Dense(16, activation='relu'),
    Dense(1, activation='______')  # 'sigmoid' (이진 분류)
])

# 모델 컴파일
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# 모델 구조 확인
model.summary()

### 문제 2 답변 작성

- 모델의 총 파라미터 수 (Total params): ______
- 파라미터 계산:
  - Dense(32): (5 + 1) × 32 = ______
  - Dense(16): (32 + 1) × 16 = ______
  - Dense(1): (16 + 1) × 1 = ______
- Dropout(0.3)의 역할: ______________________
- 출력층에 sigmoid를 사용하는 이유: ______________________

---

## 문제 3: 모델 학습 (25점)

모델을 학습하고 **EarlyStopping**을 적용하세요.

In [ ]:
# TODO: EarlyStopping 콜백 정의 (빈칸 채우기)
early_stop = EarlyStopping(
    monitor='val_loss',      # 검증 손실 모니터링
    patience=______,         # 5 에포크 동안 개선 없으면 중단
    restore_best_weights=True  # 최적 가중치로 복원
)

In [ ]:
# 모델 학습
history = model.fit(
    X_train_scaled, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,    # 학습 데이터의 20%를 검증용으로 사용
    callbacks=[early_stop],
    verbose=1
)

print(f"\n실제 학습 에포크 수: {len(history.history['loss'])}")

### 문제 3 답변 작성

- 실제 학습 에포크 수: ______
- EarlyStopping이 작동한 이유: ______________________
- `patience=5`의 의미: ______________________
- `validation_split=0.2`와 test_size=0.2의 차이점: ______________________

---

## 문제 4: 학습 곡선 분석 (15점)

**학습/검증 손실 그래프**를 그리고 과적합 여부를 진단하세요.

In [ ]:
# 학습 히스토리에서 데이터 추출
train_loss = history.history['loss']
val_loss = history.history['val_loss']
train_acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

# 손실/정확도 그래프
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: 손실 곡선
axes[0].plot(train_loss, label='Train Loss')
axes[0].plot(val_loss, label='Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 오른쪽: 정확도 곡선
axes[1].plot(train_acc, label='Train Accuracy')
axes[1].plot(val_acc, label='Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 문제 4 답변 작성

- 학습 손실(Train Loss)과 검증 손실(Val Loss)의 변화 패턴: ______________________
- 과적합의 징후가 보이는가? (학습 손실↓, 검증 손실↑이면 과적합): ______________________
- 최적의 에포크 (손실 곡선 기준): ______ 번째
- Dropout과 EarlyStopping이 과적합 방지에 효과가 있었는가?: ______________________

---

## 문제 5: 모델 평가 (15점)

테스트 데이터로 최종 모델을 평가하세요.

In [ ]:
# 테스트 데이터 평가
test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"테스트 손실: {test_loss:.4f}")
print(f"테스트 정확도: {test_accuracy:.4f}")

In [ ]:
# 예측
y_pred_prob = model.predict(X_test_scaled)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

# 분류 리포트
print("\n분류 리포트:")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Failure']))

In [ ]:
# 혼동행렬
print("\n혼동행렬:")
print(confusion_matrix(y_test, y_pred))

### 문제 5 답변 작성

- 테스트 정확도: ______
- Q3(의사결정나무)의 정확도와 비교: ______________________
- 불량 탐지 재현율 (Recall): ______
- MLP의 장점: ______________________
- MLP의 단점: ______________________

---

## 심화 문제 (선택)

### 심화 1: 모델 구조 변경 실험

In [ ]:
# 더 깊은 모델 실험
model_deep = Sequential([
    Dense(64, activation='relu', input_shape=(5,)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])
model_deep.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_deep.summary()

### 심화 2: 클래스 가중치 적용

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# 클래스 가중치 계산
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}
print(f"클래스 가중치: {class_weight_dict}")

---

## 과제 완료 체크리스트

- [ ] 문제 1: 데이터 정규화 및 이유 설명
- [ ] 문제 2: MLP 모델 설계 및 파라미터 계산
- [ ] 문제 3: 모델 학습 및 EarlyStopping 적용
- [ ] 문제 4: 학습 곡선 분석 및 과적합 진단
- [ ] 문제 5: 테스트 평가 및 모델 비교
- [ ] 모든 답변 작성 완료